# 🤝 AgentTool - Agent Composition Pattern

```mermaid
%%{init: {'theme':'base', 'themeVariables': { 'primaryColor':'#E74C3C', 'primaryTextColor':'#fff', 'primaryBorderColor':'#C0392B', 'lineColor':'#F39C12', 'secondaryColor':'#3498DB', 'tertiaryColor':'#27AE60', 'fontSize':'16px'}}}%%
graph TB
    A[👤 User Query] --> B[🤖 Main Agent]
    B --> C{🤝 AgentTool}
    C --> D[🤖 Sub-Agent 1<br/>Data Retrieval]
    C --> E[🤖 Sub-Agent 2<br/>Analysis]
    D --> F[📊 Data]
    E --> F
    F --> G[📤 Combined Results]
    
    style A fill:#3498DB,stroke:#2980B9,color:#fff
    style C fill:#E74C3C,stroke:#C0392B,color:#fff
    style D fill:#9B59B6,stroke:#8E44AD,color:#fff
    style E fill:#9B59B6,stroke:#8E44AD,color:#fff
    style G fill:#27AE60,stroke:#229954,color:#fff
```

## 📚 Learning Objectives

In this notebook, you'll learn:
1. ✅ How to use **AgentTool** for agent composition
2. ✅ Building **multi-agent systems** with specialized sub-agents
3. ✅ Creating **hierarchical agent architectures**
4. ✅ **Delegating tasks** to specialized agents
5. ✅ Best practices for agent orchestration

---

## 🎯 What is AgentTool?

**AgentTool** enables an agent to call other agents, allowing you to build complex multi-agent systems where:
- 🏗️ **Modular Design**: Break complex tasks into specialized agents
- 🎯 **Separation of Concerns**: Each agent handles specific domain
- 🔄 **Reusability**: Sub-agents can be used by multiple parent agents
- 📊 **Scalability**: Build hierarchical agent architectures

**Key Features**:
- Call registered agents by their ID
- Pass parameters between agents
- Chain agent outputs
- Build complex workflows

---

## Step 1: Import Required Libraries

In [9]:
import sys
import json

sys.path.append('..')
from agent_helpers import (
    get_os_client,
    configure_cluster_for_openai,
    create_openai_connector,
    register_and_deploy_openai_model,
    create_flow_agent,
    execute_agent,
    cleanup_resources
)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 2: Initialize OpenSearch Client

In [10]:
client = get_os_client()
info = client.info()
print(f"✅ Connected to OpenSearch: {info['cluster_name']}")
print(f"📊 Version: {info['version']['number']}")

✅ Connected to OpenSearch: docker-cluster
📊 Version: 3.3.0


## Step 3: Setup OpenAI Model

In [11]:
configure_cluster_for_openai(client)
connector_id = create_openai_connector(client)
model_id = register_and_deploy_openai_model(client, connector_id)
print(f"✅ OpenAI model ready: {model_id}")

   Configuring cluster settings for OpenAI connector...
   ✓ Cluster settings configured successfully
   Creating OpenAI connector for gpt-4o-mini...
   ✓ Connector created: 04FNa5oBAjDPEnaCLpZf
   Creating model group...
   ✓ Model group created: 1YFNa5oBAjDPEnaCLpbS
   Registering gpt-4o-mini model...
   ✓ Connector created: 04FNa5oBAjDPEnaCLpZf
   Creating model group...
   ✓ Model group created: 1YFNa5oBAjDPEnaCLpbS
   Registering gpt-4o-mini model...
   ✓ Model registered: 2YFNa5oBAjDPEnaCL5by
   Deploying model...
   ⏳ Waiting for model deployment...
      Model status: DEPLOYING
   ✓ Model registered: 2YFNa5oBAjDPEnaCL5by
   Deploying model...
   ⏳ Waiting for model deployment...
      Model status: DEPLOYING
      Model status: DEPLOYED
      ✓ Model deployed successfully!
✅ OpenAI model ready: 2YFNa5oBAjDPEnaCL5by
      Model status: DEPLOYED
      ✓ Model deployed successfully!
✅ OpenAI model ready: 2YFNa5oBAjDPEnaCL5by


## Step 4: Create Specialized Sub-Agents

Let's create specialized agents that we'll compose together.

In [12]:
# Sub-Agent 1: Data Analyst Agent
analyst_tools = [{
    "type": "MLModelTool",
    "parameters": {
        "model_id": model_id,
        "prompt": """You are a data analyst. Analyze the following data and provide insights.
        
Data: ${parameters.data:-No data provided}

Provide:
1. Key trends
2. Notable patterns
3. Recommendations

Analysis:"""
    }
}]

analyst_agent_id = create_flow_agent(
    client, "Data_Analyst_Agent",
    "Specialized agent for data analysis",
    analyst_tools
)
print(f"✅ Analyst agent created: {analyst_agent_id}")

# Sub-Agent 2: Report Writer Agent
writer_tools = [{
    "type": "MLModelTool",
    "parameters": {
        "model_id": model_id,
        "prompt": """You are a professional report writer. Create an executive summary.
        
Analysis: ${parameters.analysis:-No analysis provided}

Write a concise executive summary with:
- Overview (2-3 sentences)
- Key findings (bullet points)
- Action items

Executive Summary:"""
    }
}]

writer_agent_id = create_flow_agent(
    client, "Report_Writer_Agent",
    "Specialized agent for report writing",
    writer_tools
)
print(f"✅ Writer agent created: {writer_agent_id}")

   Registering flow agent: Data_Analyst_Agent...
   ✓ Agent registered: 3IFNa5oBAjDPEnaCT5Zg
✅ Analyst agent created: 3IFNa5oBAjDPEnaCT5Zg
   Registering flow agent: Report_Writer_Agent...
   ✓ Agent registered: 3YFNa5oBAjDPEnaCT5Zt
✅ Writer agent created: 3YFNa5oBAjDPEnaCT5Zt


## Step 5: Create Orchestrator Agents with AgentTool

Create orchestrator agents that delegate to sub-agents. **Important**: Each orchestrator can only have one AgentTool, so we create separate orchestrators for each sub-agent.

**Note**: AgentTool forwards parameters automatically, but in some OpenSearch versions, there may be limitations with parameter passing. We'll test both direct agent calls and orchestrator calls.

In [13]:
# Create orchestrator agents (one AgentTool per orchestrator)
# Orchestrator 1: Calls the analyst agent
analyst_orchestrator_tools = [{
    "type": "AgentTool",
    "parameters": {
        "agent_id": analyst_agent_id
    }
}]

analyst_orchestrator_id = create_flow_agent(
    client, "Analyst_Orchestrator",
    "Orchestrator that delegates to data analyst agent",
    analyst_orchestrator_tools
)
print(f"✅ Analyst orchestrator created: {analyst_orchestrator_id}")

# Orchestrator 2: Calls the writer agent
writer_orchestrator_tools = [{
    "type": "AgentTool",
    "parameters": {
        "agent_id": writer_agent_id
    }
}]

writer_orchestrator_id = create_flow_agent(
    client, "Writer_Orchestrator",
    "Orchestrator that delegates to report writer agent",
    writer_orchestrator_tools
)
print(f"✅ Writer orchestrator created: {writer_orchestrator_id}")

print(f"\n💡 Agents can now be chained: Orchestrator -> Sub-Agent -> Result")

   Registering flow agent: Analyst_Orchestrator...
   ✓ Agent registered: 3oFNa5oBAjDPEnaCY5aT
✅ Analyst orchestrator created: 3oFNa5oBAjDPEnaCY5aT
   Registering flow agent: Writer_Orchestrator...
   ✓ Agent registered: 34FNa5oBAjDPEnaCY5al
✅ Writer orchestrator created: 34FNa5oBAjDPEnaCY5al

💡 Agents can now be chained: Orchestrator -> Sub-Agent -> Result


## Step 4b: Verify Model is Ready

Let's verify the model is actually deployed before creating agents.

In [14]:
# Verify model status
try:
    model_status = client.transport.perform_request('GET', f'/_plugins/_ml/models/{model_id}')
    print(f"✅ Model Status: {model_status['model_state']}")
    print(f"   Model ID: {model_id}")
    
    if model_status['model_state'] != 'DEPLOYED':
        print("⚠️  Model not deployed! Waiting for deployment...")
        from agent_helpers import wait_for_model_deployment
        wait_for_model_deployment(client, model_id)
        print("✅ Model is now deployed")
except Exception as e:
    print(f"❌ Error checking model: {e}")

✅ Model Status: DEPLOYED
   Model ID: 2YFNa5oBAjDPEnaCL5by


## Step 4c: Clean Up Any Existing Agents (Optional)

If you've run this notebook before, old agents might conflict. Let's clean them up.

In [15]:
# List and optionally delete old agents with similar names
try:
    agents_response = client.transport.perform_request('GET', '/_plugins/_ml/agents/_search', 
                                                       body={"query": {"match_all": {}}})
    
    if agents_response.get('hits', {}).get('hits'):
        print("📋 Existing agents:")
        for hit in agents_response['hits']['hits']:
            agent_name = hit['_source'].get('name', 'Unknown')
            agent_id = hit['_id']
            print(f"   - {agent_name} ({agent_id})")
            
            # Uncomment to delete agents with our target names
            # if agent_name in ['Data_Analyst_Agent', 'Report_Writer_Agent', 
            #                   'Analyst_Orchestrator', 'Writer_Orchestrator']:
            #     try:
            #         client.transport.perform_request('DELETE', f'/_plugins/_ml/agents/{agent_id}')
            #         print(f"     ✓ Deleted: {agent_name}")
            #     except:
            #         pass
    else:
        print("✅ No existing agents found")
except Exception as e:
    print(f"⚠️  Could not list agents: {e}")

📋 Existing agents:
   - Data_Analyst_Agent (3IFNa5oBAjDPEnaCT5Zg)
   - Report_Writer_Agent (3YFNa5oBAjDPEnaCT5Zt)
   - Analyst_Orchestrator (3oFNa5oBAjDPEnaCY5aT)
   - Writer_Orchestrator (34FNa5oBAjDPEnaCY5al)


## Step 6: Test Case 1 - Sales Data Analysis (Direct Agent Call)

First, let's test the analyst agent directly to ensure it works.

In [16]:
# Test the analyst agent directly first
sales_data = """Sales Data Q1-Q3 2025:
Q1: $2.5M revenue, 15% growth
Q2: $2.8M revenue, 12% growth
Q3: $2.4M revenue, -14% decline

Customer metrics:
- New customers: 500, 450, 350
- Churn rate: 5%, 6%, 9%"""

parameters = {"data": sales_data}

print("📊 Testing Analyst Agent Directly...")
print("="*60)
analysis_response = execute_agent(client, analyst_agent_id, parameters)
print(json.dumps(analysis_response, indent=2))

print("\n\n📊 Now testing via Orchestrator -> Analyst Agent...")
print("="*60)
# The orchestrator will forward the same parameters
orchestrator_response = execute_agent(client, analyst_orchestrator_id, parameters)
print(json.dumps(orchestrator_response, indent=2))

📊 Testing Analyst Agent Directly...


RequestError: RequestError(400, 'IllegalArgumentException')

## Step 6a: Debug - Test Simple Agent First

Let's create a very simple agent first to ensure the basic setup works.

In [ ]:
# Create a very simple test agent with minimal prompt
test_tools = [{
    "type": "MLModelTool",
    "parameters": {
        "model_id": model_id,
        "prompt": "You are a helpful assistant. Answer this: ${parameters.question}"
    }
}]

test_agent_id = create_flow_agent(
    client, "Simple_Test_Agent",
    "A simple test agent",
    test_tools
)

# Test with simple parameters
test_params = {"question": "What is 2+2?"}
print("🧪 Testing simple agent...")
try:
    test_response = execute_agent(client, test_agent_id, test_params)
    print("✅ Simple agent works!")
    print(json.dumps(test_response, indent=2))
except Exception as e:
    print(f"❌ Error: {e}")

## Step 7: Test Case 2 - Generate Executive Report (Direct and via Orchestrator)

In [ ]:
# Extract analysis from response and send to writer agent
analysis_text = "Revenue grew in Q1-Q2 but declined in Q3. Customer churn increased significantly."

parameters = {"analysis": analysis_text}

print("📝 Testing Writer Agent Directly...")
print("="*60)
report_response = execute_agent(client, writer_agent_id, parameters)
print(json.dumps(report_response, indent=2))

print("\n\n📝 Now testing via Orchestrator -> Writer Agent...")
print("="*60)
orchestrator_response = execute_agent(client, writer_orchestrator_id, parameters)
print(json.dumps(orchestrator_response, indent=2))

## Step 8: Create Simple Delegation Example

In [ ]:
# Create a simple math agent
math_tools = [{
    "type": "MLModelTool",
    "parameters": {
        "model_id": model_id,
        "prompt": "Solve this math problem: ${parameters.question}\n\nSolution:"
    }
}]

math_agent_id = create_flow_agent(
    client, "Math_Agent",
    "Specialized agent for mathematical calculations",
    math_tools
)

# Create delegator agent
delegator_tools = [{
    "type": "AgentTool",
    "parameters": {
        "agent_id": math_agent_id
    }
}]

delegator_id = create_flow_agent(
    client, "Delegator_Agent",
    "Agent that delegates math problems to math specialist",
    delegator_tools
)

print(f"✅ Math delegation system created")

# Test delegation
parameters = {"question": "If revenue grew from $2M to $2.5M, what's the percentage increase?"}
print("\n❓ Math Question:")
print(parameters["question"])
print("="*60)
response = execute_agent(client, delegator_id, parameters)
print(json.dumps(response, indent=2))

## 🎓 Key Takeaways

### What We Learned:

1. **AgentTool Capabilities**:
   - ✅ Enable agent-to-agent communication
   - ✅ Build hierarchical agent systems
   - ✅ Create specialized, reusable agents
   - ✅ Delegate tasks to expert agents
   - ⚠️ **Limitation**: Each agent can only have ONE AgentTool (no duplicates)

2. **Multi-Agent Patterns**:
   ```
   # Pattern 1: One orchestrator per sub-agent
   Orchestrator A -> Sub-Agent 1 (Specialist A)
   Orchestrator B -> Sub-Agent 2 (Specialist B)
   Orchestrator C -> Sub-Agent 3 (Specialist C)
   
   # Pattern 2: Hierarchical chain
   Main Orchestrator -> Manager Agent -> Worker Agents
   ```

3. **Use Cases**:
   - 🎯 **Task Decomposition**: Break complex tasks into steps
   - 🏗️ **Domain Specialization**: Each agent masters one domain
   - 🔄 **Workflow Orchestration**: Chain agent outputs
   - 📊 **Parallel Processing**: Multiple agents work simultaneously

4. **Architecture Patterns**:
   ```python
   # Sequential workflow (chain multiple orchestrators)
   Orchestrator1 -> SubAgent1 -> Orchestrator2 -> SubAgent2 -> Result
   
   # Parallel execution (call multiple orchestrators independently)
   [Orchestrator1 -> SubAgent1, Orchestrator2 -> SubAgent2] -> Combine
   
   # Hierarchical (orchestrators calling other orchestrators)
   Main Orchestrator -> Manager Orchestrator -> Worker Agent
   ```

5. **Important Constraint**:
   - ❌ **Cannot do**: One agent with multiple AgentTools
     ```python
     # This causes "Duplicate tool defined" error
     tools = [
         {"type": "AgentTool", "parameters": {"agent_id": "agent1"}},
         {"type": "AgentTool", "parameters": {"agent_id": "agent2"}}  # Error!
     ]
     ```
   - ✅ **Must do**: Separate orchestrator for each sub-agent
     ```python
     # Orchestrator A calls agent1
     orchestrator_a_tools = [{"type": "AgentTool", "parameters": {"agent_id": "agent1"}}]
     # Orchestrator B calls agent2
     orchestrator_b_tools = [{"type": "AgentTool", "parameters": {"agent_id": "agent2"}}]
     ```

### Best Practices:

- ✅ **Single Responsibility**: Each sub-agent has one clear purpose
- ✅ **One AgentTool Per Orchestrator**: Create separate orchestrators for different sub-agents
- ✅ **Loose Coupling**: Agents communicate through defined interfaces
- ✅ **Reusability**: Design sub-agents for multiple contexts
- ✅ **Error Handling**: Implement fallbacks for failed agents
- ✅ **Testing**: Test sub-agents independently first
- ✅ **Orchestrator Naming**: Use clear names like "Analyst_Orchestrator" to show delegation purpose

### Design Principles:

1. **Modularity**: Break complex systems into simple agents
2. **Specialization**: Each agent excels at specific tasks
3. **Composition**: Combine simple agents for complex behaviors
4. **Scalability**: Add new agents without changing existing ones

### Example Architectures:

```python
# Customer Service System (Multiple Orchestrators)
Intent_Orchestrator -> Intent_Classifier_Agent
Product_Orchestrator -> Product_Info_Agent
Order_Orchestrator -> Order_Status_Agent
Escalation_Orchestrator -> Escalation_Agent

# Data Pipeline (Sequential Orchestrators)
Collector_Orchestrator -> Data_Collector_Agent
   └─> Validator_Orchestrator -> Data_Validator_Agent
          └─> Transformer_Orchestrator -> Data_Transformer_Agent
                └─> Publisher_Orchestrator -> Data_Publisher_Agent

# Research Assistant (Parallel Orchestrators)
[WebSearch_Orchestrator -> Web_Search_Agent,
 DocAnalysis_Orchestrator -> Document_Analysis_Agent] 
   └─> Summary_Orchestrator -> Summary_Agent
         └─> Citation_Orchestrator -> Citation_Agent
```

### Real Implementation Example:

```python
# Create specialized sub-agents
analyst_agent = create_flow_agent(client, "Analyst", "Analyzes data", [MLModelTool])
writer_agent = create_flow_agent(client, "Writer", "Writes reports", [MLModelTool])

# Create orchestrators (one per sub-agent)
analyst_orch = create_flow_agent(client, "Analyst_Orchestrator", 
    "Delegates to analyst", [{"type": "AgentTool", "agent_id": analyst_agent}])

writer_orch = create_flow_agent(client, "Writer_Orchestrator",
    "Delegates to writer", [{"type": "AgentTool", "agent_id": writer_agent}])

# Use them in sequence
data = {"input": "sales data"}
analysis = execute_agent(client, analyst_orch, data)  # Step 1
report = execute_agent(client, writer_orch, analysis)  # Step 2
```

















---```report = execute_agent(client, writer_orch, analysis)  # Step 2analysis = execute_agent(client, analyst_orch, data)  # Step 1data = {"input": "sales data"}# Use them in sequence    "Delegates to writer", [{"type": "AgentTool", "agent_id": writer_agent}])writer_orch = create_flow_agent(client, "Writer_Orchestrator",    "Delegates to analyst", [{"type": "AgentTool", "agent_id": analyst_agent}])analyst_orch = create_flow_agent(client, "Analyst_Orchestrator", # Create orchestrators (one per sub-agent)writer_agent = create_flow_agent(client, "Writer", "Writes reports", [MLModelTool])analyst_agent = create_flow_agent(client, "Analyst", "Analyzes data", [MLModelTool])

---















---```└── Citation Agent├── Summary Agent├── Document Analysis Agent├── Web Search AgentMain Research Agent# Research Assistant└── Data Publisher Agent├── Data Transformer Agent├── Data Validator Agent├── Data Collector AgentOrchestrator# Data Pipeline└── Citation Agent
```

---

## 🧹 Cleanup (Optional)

In [ ]:
# # Cleanup all agents
# cleanup_resources(
#     client=client,
#     agent_ids=[
#         analyst_orchestrator_id,
#         writer_orchestrator_id,
#         analyst_agent_id,
#         writer_agent_id,
#         math_agent_id,
#         delegator_id
#     ],
#     model_ids=[model_id],
#     connector_ids=[connector_id]
# )
# print("✅ Cleanup complete!")

## 🚀 Next Steps

- **ScratchpadTools**: Add memory to multi-agent systems
- **RAGTool**: Combine retrieval with agent composition
- **QueryPlanningTool**: Build query generation pipelines

---

📚 **Resources**:
- [ML Commons Agent Tools](https://opensearch.org/docs/latest/ml-commons-plugin/agents-tools/)
- [AgentTool Documentation](https://opensearch.org/docs/latest/ml-commons-plugin/agents-tools/tools/agent-tool/)